# Desafio Transparência - Análise Consolidada (Camada Gold)
Este notebook realiza a extração, transformação e análise de dados sobre diárias, passagens e viagens corporativas. Os resultados são apresentados em DataFrames e exportados como imagens estáticas em alta resolução (`.png`) na pasta `graficos_png/`.

In [4]:
import sys
import os
import pandas as pd
import plotly.express as px
import plotly.io as pio

# Garantir que a pasta de gráficos exista
os.makedirs('graficos_png', exist_ok=True)

# Definir renderizador estático como padrão para visualização limpa e exportação
pio.renderers.default = 'png'

# Conexão com o banco de dados (Importando diretamente do banco.py da raiz)
from banco import conectar

conn = conectar()
print('Conexão estabelecida com sucesso!')

Conexão estabelecida com sucesso!


## 1. Top 5 Órgãos com Maior Custo Total
**Pergunta de Negócio:** Quais são os 5 órgãos públicos que acumularam o maior valor em gastos com viagens e diárias?

In [5]:
query_1 = '''
SELECT 
    `Nome do órgão superior` AS orgao,
    SUM(`Valor total da viagem`) AS custo_total
FROM gold_viagens_consolidadas
GROUP BY 1
ORDER BY custo_total DESC
LIMIT 5;
'''
df1 = pd.read_sql(query_1, conn)
df1_sorted = df1.sort_values('custo_total', ascending=True)

fig1 = px.bar(
    df1_sorted,
    x='custo_total',
    y='orgao',
    orientation='h',
    text=df1_sorted['custo_total'].apply(lambda x: f'R$ {x:,.2f}'.replace(',', 'X').replace('.', ',').replace('X', '.').strip()),
    title='Top 5 Órgãos por Custo Total de Viagens'
)

max_val1 = df1_sorted['custo_total'].max()
fig1.update_traces(
    textposition='outside',
    texttemplate='  %{text}',
    cliponaxis=False
)
fig1.update_layout(
    margin=dict(l=260, r=180, t=60, b=50),
    xaxis=dict(range=[0, max_val1 * 1.35], title='Custo Total (R$)'),
    yaxis=dict(title='')
)

fig1.write_image('graficos_png/1_maior_custo_total_orgao.png', scale=2)
fig1.show()

/tmp/ipykernel_15945/1691655652.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df1 = pd.read_sql(query_1, conn)


ProgrammingError: 1146 (42S02): Table 'transparencia.gold_viagens_consolidadas' doesn't exist

## 2. Top 3 Destinos com Maior Custo Médio por Viagem
**Pergunta de Negócio:** Quais destinos possuem a maior média de custo por ocorrência individual de viagem?

In [ ]:
query_2 = '''
SELECT 
    `Destinos` AS destino,
    AVG(`Valor total da viagem`) AS custo_medio
FROM gold_viagens_consolidadas
WHERE `Destinos` IS NOT NULL AND `Destinos` != ''
GROUP BY 1
ORDER BY custo_medio DESC
LIMIT 3;
'''
df2 = pd.read_sql(query_2, conn)

fig2 = px.bar(
    df2,
    x='destino',
    y='custo_medio',
    text=df2['custo_medio'].apply(lambda x: f'R$ {x:,.2f}'.replace(',', 'X').replace('.', ',').replace('X', '.').strip()),
    title='Top 3 Destinos com Maior Custo Médio'
)

max_val2 = df2['custo_medio'].max()
fig2.update_traces(
    textposition='outside',
    cliponaxis=False
)
fig2.update_layout(
    margin=dict(l=60, r=60, t=80, b=60),
    yaxis=dict(range=[0, max_val2 * 1.35], title='Custo Médio (R$)'),
    xaxis=dict(title='Destino')
)

fig2.write_image('graficos_png/2_custo_medio_destino.png', scale=2)
fig2.show()

## 3. Viagem de Maior Duração
**Pergunta de Negócio:** Qual foi o registro de viagem corporativa com o maior número de dias contínuos e qual foi o valor total associado?

In [ ]:
query_3 = '''
SELECT 
    `Identificador do processo de viagem` AS id_viagem,
    `Nome` AS viajante,
    `Motivo do afastamento` AS motivo,
    `Duração em Dias` AS duracao_dias,
    `Valor total da viagem` AS custo_total
FROM gold_viagens_consolidadas
ORDER BY duracao_dias DESC
LIMIT 1;
'''
df3 = pd.read_sql(query_3, conn)
display(df3)

## 4. Tipo de Pagamento com Maior Valor Médio
**Pergunta de Negócio:** Qual a modalidade financeira/tipo de pagamento que apresenta o maior ticket médio transacionado?

In [ ]:
query_4 = '''
SELECT 
    `Tipo de pagamento` AS tipo_pagamento,
    AVG(`Valor`) AS valor_medio
FROM gold_pagamento_limpo
WHERE `Tipo de pagamento` IS NOT NULL AND `Tipo de pagamento` != ''
GROUP BY 1
ORDER BY valor_medio DESC;
'''
df4 = pd.read_sql(query_4, conn)

fig4 = px.bar(
    df4,
    x='tipo_pagamento',
    y='valor_medio',
    text=df4['valor_medio'].apply(lambda x: f'R$ {x:,.2f}'.replace(',', 'X').replace('.', ',').replace('X', '.').strip()),
    title='Valor Médio por Tipo de Pagamento'
)

max_val4 = df4['valor_medio'].max()
fig4.update_traces(
    textposition='outside',
    cliponaxis=False
)
fig4.update_layout(
    margin=dict(l=60, r=60, t=80, b=60),
    yaxis=dict(range=[0, max_val4 * 1.35], title='Valor Médio (R$)'),
    xaxis=dict(title='Tipo de Pagamento')
)

fig4.write_image('graficos_png/4_valor_medio_pagamento.png', scale=2)
fig4.show()

## 5. Meio de Transporte Mais Utilizado
**Pergunta de Negócio:** Qual a proporção e distribuição do uso de meios de transporte nos trechos realizados?

In [ ]:
query_5 = '''
SELECT 
    `Meio de transporte` AS meio_transporte,
    COUNT(*) AS total_trechos
FROM gold_trecho_limpo
WHERE `Meio de transporte` IS NOT NULL AND `Meio de transporte` != ''
GROUP BY 1
ORDER BY total_trechos DESC;
'''
df5 = pd.read_sql(query_5, conn)

fig5 = px.pie(
    df5,
    values='total_trechos',
    names='meio_transporte',
    title='Distribuição dos Meios de Transporte Utilizados',
    hole=0.4
)
fig5.update_traces(textinfo='percent+label')
fig5.update_layout(margin=dict(l=50, r=50, t=60, b=50))

fig5.write_image('graficos_png/5_meio_transporte_mais_usado.png', scale=2)
fig5.show()

## 6. Frequência de Trechos por UF de Destino
**Pergunta de Negócio:** Quais estados da federação (UF) concentram o maior volume de trechos em viagens corporativas?

In [ ]:
query_6 = '''
SELECT 
    `UF - Destino` AS uf_destino,
    COUNT(*) AS frequencia
FROM gold_trecho_limpo
WHERE `UF - Destino` IS NOT NULL AND `UF - Destino` != ''
GROUP BY 1
ORDER BY frequencia DESC;
'''
df6 = pd.read_sql(query_6, conn)

fig6 = px.bar(
    df6,
    x='uf_destino',
    y='frequencia',
    text='frequencia',
    title='Frequência de Trechos por UF de Destino'
)

max_val6 = df6['frequencia'].max()
fig6.update_traces(
    textposition='outside',
    cliponaxis=False
)
fig6.update_layout(
    margin=dict(l=50, r=50, t=80, b=100),
    xaxis=dict(tickangle=-45, title='UF de Destino'),
    yaxis=dict(range=[0, max_val6 * 1.3], title='Total de Trechos')
)

fig6.write_image('graficos_png/6_uf_destino_frequencia.png', scale=2)
fig6.show()

## 7. Órgão Superior que Efetivamente Pagou Mais no Total
**Pergunta de Negócio:** Qual unidade gestora realizou a maior soma em pagamentos liquidados/efetivados?

In [ ]:
query_7 = '''
SELECT 
    v.`Nome do órgão superior` AS orgao,
    SUM(p.`Valor`) AS total_pago
FROM gold_pagamento_limpo p
JOIN gold_viagens_consolidadas v 
  ON p.`Identificador do processo de viagem` = v.`Identificador do processo de viagem`
WHERE v.`Nome do órgão superior` IS NOT NULL
GROUP BY 1
ORDER BY total_pago DESC
LIMIT 5;
'''
df7 = pd.read_sql(query_7, conn)
df7_sorted = df7.sort_values('total_pago', ascending=True)

fig7 = px.bar(
    df7_sorted,
    x='total_pago',
    y='orgao',
    orientation='h',
    text=df7_sorted['total_pago'].apply(lambda x: f'R$ {x:,.2f}'.replace(',', 'X').replace('.', ',').replace('X', '.').strip()),
    title='Top Órgãos por Total Efetivamente Pago'
)

max_val7 = df7_sorted['total_pago'].max()
fig7.update_traces(
    textposition='outside',
    texttemplate='  %{text}',
    cliponaxis=False
)
fig7.update_layout(
    margin=dict(l=260, r=180, t=60, b=50),
    xaxis=dict(range=[0, max_val7 * 1.35], title='Total Pago (R$)'),
    yaxis=dict(title='')
)

fig7.write_image('graficos_png/7_orgao_que_mais_pagou.png', scale=2)
fig7.show()